<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/03_AI_Engineer/intermediate/01_advanced_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced RAG Techniques

Move beyond naive RAG — hybrid search, reranking, HyDE, and multi-query strategies for production-grade retrieval quality.

**Topics:** Hybrid search (BM25 + semantic), cross-encoder reranking, HyDE, parent-child chunking, query expansion, FLARE

## 1. Hybrid Search — BM25 + Dense Retrieval

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from dataclasses import dataclass
from typing import Optional

@dataclass
class RetrievedDoc:
    text: str
    source: str
    bm25_score: float = 0.0
    semantic_score: float = 0.0
    hybrid_score: float = 0.0

class HybridRetriever:
    """Combines BM25 (keyword) and dense (semantic) retrieval with RRF fusion.
    
    BM25 excels at: exact keyword matches, rare terms, product codes
    Dense excels at: semantic similarity, paraphrasing, concept matching
    Hybrid: best of both worlds — ~15-20% better recall than either alone
    """
    
    def __init__(self, embed_model: str = 'all-MiniLM-L6-v2'):
        self.embed_model = SentenceTransformer(embed_model)
        self.documents: list[str] = []
        self.bm25: Optional[BM25Okapi] = None
        self.embeddings: Optional[np.ndarray] = None
    
    def index(self, documents: list[str]):
        self.documents = documents
        tokenized = [doc.lower().split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized)
        self.embeddings = self.embed_model.encode(documents, convert_to_numpy=True)
        self.embeddings /= np.linalg.norm(self.embeddings, axis=1, keepdims=True)  # L2 normalize
    
    def retrieve(self, query: str, top_k: int = 5, alpha: float = 0.5) -> list[RetrievedDoc]:
        """alpha=0: pure BM25, alpha=1: pure dense, alpha=0.5: balanced hybrid."""
        # BM25 scores
        bm25_scores = np.array(self.bm25.get_scores(query.lower().split()))
        bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-8)
        
        # Dense scores
        q_emb = self.embed_model.encode([query], convert_to_numpy=True)
        q_emb /= np.linalg.norm(q_emb)
        dense_scores = (self.embeddings @ q_emb.T).squeeze()
        dense_norm = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min() + 1e-8)
        
        # Hybrid fusion
        hybrid_scores = (1 - alpha) * bm25_norm + alpha * dense_norm
        top_indices = np.argsort(hybrid_scores)[::-1][:top_k]
        
        return [RetrievedDoc(
            text=self.documents[i], source=f'doc_{i}',
            bm25_score=float(bm25_norm[i]), semantic_score=float(dense_norm[i]),
            hybrid_score=float(hybrid_scores[i]),
        ) for i in top_indices]

print('Hybrid retrieval strategy:')
print('  alpha=0.0: pure BM25 (best for keyword queries)')
print('  alpha=0.5: balanced (recommended default)')
print('  alpha=1.0: pure dense (best for semantic queries)')
print()
print('Reciprocal Rank Fusion (RRF) is an alternative to linear combination:')
print('  RRF_score(doc) = Σ 1/(k + rank_in_list_i) for each retriever i')

## 2. Cross-Encoder Reranking

In [ ]:
from sentence_transformers import CrossEncoder
import time

class RerankedRetriever:
    """Two-stage retrieval: fast bi-encoder first, slow cross-encoder reranks top-N.
    
    Why two stages?
    - Bi-encoder (fast): embeds query and docs independently → compare with dot product
    - Cross-encoder (slow): sees query+doc together → much better relevance scoring
    - Strategy: retrieve top-50 with bi-encoder, rerank to top-5 with cross-encoder
    """
    
    def __init__(self, retriever: HybridRetriever, reranker_model: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2'):
        self.retriever = retriever
        self.reranker = CrossEncoder(reranker_model)
    
    def retrieve_and_rerank(
        self,
        query: str,
        initial_k: int = 20,
        final_k: int = 5,
    ) -> list[RetrievedDoc]:
        # Stage 1: Fast retrieval (bi-encoder)
        candidates = self.retriever.retrieve(query, top_k=initial_k)
        
        # Stage 2: Slow reranking (cross-encoder)
        pairs = [(query, doc.text) for doc in candidates]
        rerank_scores = self.reranker.predict(pairs)
        
        # Sort by cross-encoder score
        ranked = sorted(
            zip(candidates, rerank_scores),
            key=lambda x: x[1], reverse=True,
        )
        return [doc for doc, _ in ranked[:final_k]]

# Latency profile of two-stage retrieval
stages = [
    ('Embedding query', '5ms', 'Bi-encoder encodes query'),
    ('Vector search (top-20)', '10ms', 'ANN search in vector store'),
    ('Cross-encoder reranking', '80ms', '20 pairs through cross-encoder'),
    ('Total', '~95ms', 'vs ~10ms for bi-encoder only'),
]
print('Two-stage retrieval latency:')
for stage, latency, note in stages:
    print(f'  {stage:<35} {latency:<8} {note}')
print()
print('Accuracy improvement:')
print('  Bi-encoder only:    NDCG@10 = 0.68')
print('  + Cross-encoder:    NDCG@10 = 0.81  (+19%)')
print('  + Hybrid retrieval: NDCG@10 = 0.85  (+25%)')

## 3. HyDE — Hypothetical Document Embeddings

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

HYDE_PROMPT = """Write a paragraph that would be the perfect answer to the following question.
Write as if you are an expert. Be specific and factual.
The paragraph will be used to search a document database.

Question: {query}

Hypothetical answer paragraph:"""

def hyde_retrieve(query: str, retriever: HybridRetriever, top_k: int = 5) -> list[RetrievedDoc]:
    """HyDE: generate a hypothetical document, use its embedding to retrieve.
    
    Why HyDE helps:
    - Queries are short, sparse signals
    - Documents are long, dense passages
    - HyDE bridges this gap by generating a document-like embedding
    """
    # Step 1: Generate hypothetical document
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': HYDE_PROMPT.format(query=query)}],
        max_tokens=200,
    )
    hypothetical_doc = response.choices[0].message.content
    
    # Step 2: Use hypothetical doc embedding (instead of query embedding) for search
    results = retriever.retrieve(hypothetical_doc, top_k=top_k, alpha=1.0)
    return results

# HyDE performance on different query types
query_types = [
    ('Factual: "What is the capital of France?"', 'Minimal gain — direct retrieval works well'),
    ('Abstract: "How does attention mechanism work?"', '+10-15% — HyDE fills conceptual gap'),
    ('Comparative: "Pros of BERT vs GPT?"', '+15-20% — generates structured comparison'),
    ('Instructional: "How to fine-tune LLaMA?"', '+20-25% — generates procedural text'),
]
print('HyDE improvement by query type:')
for query, gain in query_types:
    print(f'  {query:<45}: {gain}')
print()
print('Cost: 1 extra LLM call per query (~$0.0001 with gpt-4o-mini)')
print('Latency: +200-500ms (LLM generation time)')

## 4. Parent-Child Chunking for Better Context

In [ ]:
from dataclasses import dataclass, field

@dataclass
class ParentChunk:
    """Large chunk (512-1024 tokens) returned to the LLM for context."""
    id: str
    text: str
    source: str
    children: list[str] = field(default_factory=list)  # child chunk IDs

@dataclass
class ChildChunk:
    """Small chunk (64-128 tokens) used for embedding and retrieval."""
    id: str
    text: str
    parent_id: str  # reference to parent

def create_parent_child_chunks(
    text: str,
    source: str,
    parent_size: int = 512,   # words in parent chunk
    child_size: int = 64,     # words in child chunk
) -> tuple[list[ParentChunk], list[ChildChunk]]:
    """Parent chunks are indexed for retrieval; child chunks are returned to LLM.
    
    Why this helps:
    - Embedding a large chunk loses local detail (averaged representation)
    - Small chunks embed precisely but lack context when read by LLM
    - Solution: embed small → return large parent for LLM context
    """
    words = text.split()
    parents, children = [], []
    
    for p_idx, p_start in enumerate(range(0, len(words), parent_size)):
        parent_words = words[p_start:p_start + parent_size]
        parent_id = f'{source}_parent_{p_idx}'
        parent = ParentChunk(id=parent_id, text=' '.join(parent_words), source=source)
        
        for c_idx, c_start in enumerate(range(0, len(parent_words), child_size)):
            child_words = parent_words[c_start:c_start + child_size]
            child_id = f'{parent_id}_child_{c_idx}'
            children.append(ChildChunk(id=child_id, text=' '.join(child_words), parent_id=parent_id))
            parent.children.append(child_id)
        
        parents.append(parent)
    
    return parents, children

# Demonstrate parent-child relationship
sample_text = ' '.join(['word'] * 600)
parents, children = create_parent_child_chunks(sample_text, 'doc.txt')
print(f'Document: 600 words')
print(f'Parents (512 words each): {len(parents)}')
print(f'Children (64 words each): {len(children)}')
print(f'Children per parent: {len(parents[0].children)}')
print()
print('Retrieval flow:')
print('  1. Embed all child chunks → store in vector DB')
print('  2. Query → find top-k child chunks')
print('  3. Look up parent chunk for each child')
print('  4. Deduplicate parents → return to LLM')
print('  Result: precise retrieval + rich context')

## 5. Multi-Query Retrieval and Query Expansion

In [ ]:
MULTI_QUERY_PROMPT = """Generate {n} different phrasings of the following question.
Each phrasing should approach the question from a different angle or use different terminology.
Output each on a new line. Only output the questions, no numbering or extra text.

Original question: {query}"""

def generate_query_variants(query: str, n: int = 4) -> list[str]:
    """Generate N query reformulations to improve recall."""
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': MULTI_QUERY_PROMPT.format(query=query, n=n)}],
        temperature=0.7,
    )
    variants = response.choices[0].message.content.strip().split('\n')
    return [query] + [v.strip() for v in variants if v.strip()]

def multi_query_retrieve(
    query: str,
    retriever: HybridRetriever,
    top_k_per_query: int = 5,
) -> list[RetrievedDoc]:
    """Retrieve with multiple query reformulations, deduplicate results."""
    queries = generate_query_variants(query)
    
    seen_texts: set[str] = set()
    all_results: list[RetrievedDoc] = []
    
    for q in queries:
        results = retriever.retrieve(q, top_k=top_k_per_query)
        for doc in results:
            if doc.text not in seen_texts:
                seen_texts.add(doc.text)
                all_results.append(doc)
    
    # Re-rank combined results by average score
    return sorted(all_results, key=lambda d: d.hybrid_score, reverse=True)

# Simulated query variants
original = 'How does attention mechanism work in transformers?'
simulated_variants = [
    'How does attention mechanism work in transformers?',
    'What is self-attention and how is it computed?',
    'Explain the scaled dot-product attention formula',
    'How do transformer models learn to focus on relevant tokens?',
    'What is multi-head attention and why is it used?',
]
print(f'Original: "{original}"')
print(f'\nGenerated {len(simulated_variants)-1} variants:')
for i, v in enumerate(simulated_variants[1:], 1):
    print(f'  {i}. {v}')
print()
print(f'Result: {len(simulated_variants) * 5} candidates → deduplicated → top-5')

## Exercises

1. **Reciprocal Rank Fusion:** Implement RRF to fuse BM25 and dense retrieval results. Compare it to the linear combination (alpha weighting) on 20 test queries. Which performs better and why?

2. **Adaptive alpha:** Build a classifier that predicts the optimal alpha value (BM25 vs dense weight) based on query characteristics (length, contains exact terms, question type). Train it on a small labeled dataset.

3. **FLARE implementation:** Implement Forward-Looking Active Retrieval (FLARE) — generate text token by token, and when the model is uncertain (low token probability), pause and retrieve relevant documents before continuing.